# 03 · Train XLSR-300m cross-lingual classifier


# 03 · VoiceShield SIH — Train XLSR-300m cross-lingual classifier (Colab GPU)

Same recipe as notebook 02 but with `facebook/wav2vec2-xls-r-300m` — better
Indic-language coverage at the cost of being bigger/slower. Excellent for the
final accuracy claim in the SIH demo.

## 1 · Setup + load (same as 02)

In [ ]:
!pip install -q datasets[audio] soundfile librosa transformers evaluate accelerate

import os, numpy as np, pandas as pd

DRIVE_OK = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OK = os.path.isdir("/content/drive/MyDrive")
except Exception as exc:  # noqa: BLE001
    print("Drive mount failed — continuing:", exc)
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN", "")
except Exception:  # noqa: BLE001
    pass
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
ROOT = "/content/drive/MyDrive/voiceshield_cloud"
if DRIVE_OK and os.path.isfile(f"{ROOT}/train.csv"):
    train = pd.read_csv(f"{ROOT}/train.csv"); val = pd.read_csv(f"{ROOT}/val.csv")
    print("Loaded from Drive")
elif HF_TOKEN.startswith("hf_"):
    from datasets import load_dataset
    ds = load_dataset("VAIVE/voiceshield-sih")
    train = ds["train"].to_pandas(); val = ds["val"].to_pandas()
    print("Loaded from HF Hub")
else:
    raise SystemExit("No dataset found. Re-run notebook 01 (accept the Drive popup) or "
                     "set HF_TOKEN and push the dataset to the Hub.")
print("train:", len(train), " val:", len(val))

In [ ]:
from datasets import Dataset, Audio, ClassLabel, Features, Value
feats = Features({"audio": Audio(sampling_rate=16000),
                  "label": ClassLabel(names=["bonafide","spoof"]),
                  "language": Value("string")})
def to_ds(df):
    df = df.copy(); df["label"] = (df.label == "spoof").astype("int8")
    df = df.rename(columns={"audio_path": "audio"})
    return Dataset.from_pandas(df[["audio","label","language"]], features=feats)
train_ds, val_ds = to_ds(train), to_ds(val)

## 2 · Model + prep

In [ ]:
from transformers import (Wav2Vec2Processor, Wav2Vec2ForAudioClassification,
                          TrainingArguments, Trainer)
import evaluate, numpy as np
ckpt = "facebook/wav2vec2-xls-r-300m"
processor = Wav2Vec2Processor.from_pretrained(ckpt, do_normalize=True)
model = Wav2Vec2ForAudioClassification.from_pretrained(
    ckpt, num_labels=2, attention_dropout=0.1, hidden_dropout=0.1,
    label2id={"bonafide":0,"spoof":1}, id2label={0:"bonafide",1:"spoof"})
MAX = 16000*8
def load_audio(x):
    """Works with datasets 2.x (dict) and 3.x+torchcodec (AudioDecoder)."""
    a = x["audio"]
    if isinstance(a, dict):
        return np.asarray(a["array"], dtype=np.float32), a.get("sampling_rate", 16000)
    s = a.get_all_samples()
    arr = np.asarray(s.data.cpu() if hasattr(s.data, "cpu") else s.data,
                     dtype=np.float32).squeeze()
    if arr.ndim > 1:
        arr = arr.mean(axis=0)
    return arr, s.sample_rate

def pre(x):
    arr, sr = load_audio(x)
    arr = np.pad(arr, (0, max(0, MAX-len(arr))))[:MAX]
    return processor(arr, sampling_rate=sr).input_values[0]
train_ds = train_ds.map(lambda x: {"input_values": pre(x)}, remove_columns=["audio"])
val_ds   = val_ds.map(lambda x: {"input_values": pre(x)}, remove_columns=["audio"])

## 3 · Train (smaller batch; XLS-R is heavy)

In [ ]:
args = TrainingArguments(
    output_dir="/content/vs_out_xlsr",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to=[],
    fp16=True,
    metric_for_best_model="eval_accuracy",
    load_best_model_at_end=True,
)
acc = evaluate.load("accuracy")
trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=val_ds,
                  compute_metrics=lambda p: acc.compute(
                      predictions=p.predictions.argmax(-1), references=p.label_ids))
trainer.train()

## 4 · Save + push

In [ ]:
SAVE = ROOT if DRIVE_OK else "/content/vs_model_xlsr"
model.save_pretrained(SAVE)
processor.save_pretrained(SAVE)
print("saved →", SAVE)

if HF_TOKEN.startswith("hf_"):
    model.push_to_hub("VAIVE/voiceshield-xlsr")
    processor.push_to_hub("VAIVE/voiceshield-xlsr")
print("DONE")